# 01 — Load the oracle and run a query

This notebook is the shortest path from a fresh Remanentia
install to a working retrieval pipeline. It runs **offline** on
two synthetic sessions, then optionally loads the LongMemEval
oracle if you have obtained it separately.

What you will see:

1. Decompose conversation sessions into `AtomicFact` records.
2. Build the `ArcaneRetriever` over those facts.
3. Run a query, show ranked facts plus an extracted answer.
4. Optional cell that repeats the exercise on a real oracle item.

Dependencies: `pip install -e .[all]` in the repo root. No
OpenAI / Anthropic key required for the rule-based path used
here.

© 1996–2026 Miroslav Šotek. AGPL-3.0-or-later.

## 1. Synthetic sessions

Two chat sessions with overlapping entities. Keeping the
example small so the notebook stays fast to run.

In [ ]:
sessions = [
    [
        {"role": "user", "content": "I shipped the STDP fix for sc-neurocore on March 15, 2026."},
        {"role": "assistant", "content": "Nice — did the LOCOMO score improve?"},
        {"role": "user", "content": "LOCOMO went from 81.2% to 88.5% after the fix."},
    ],
    [
        {"role": "user", "content": "On March 22 I tagged remanentia v3.14 and published."},
        {"role": "assistant", "content": "And the next tag?"},
        {"role": "user", "content": "Skipping 3.15 for now. The BM25 change is in 3.16."},
    ],
]

session_dates = ["2026-03-15", "2026-03-22"]
len(sessions), len(sessions[0]), len(sessions[1])

## 2. Decompose into atomic facts

`fact_decomposer.decompose_sessions` splits each turn into one
or more `AtomicFact` records, tagging each with an entity set
and a `session_date` anchor.

In [ ]:
from fact_decomposer import decompose_sessions

facts = decompose_sessions(sessions, session_dates=session_dates)
print(f"{len(facts)} atomic facts\n")
for f in facts[:5]:
    entities = ", ".join(sorted(f.entities)[:3]) if f.entities else "-"
    print(f"  [{f.session_date}] {entities:<20} {f.text[:80]}")

## 3. Build the retriever, run a query

`ArcaneRetriever` runs four parallel channels (FAST, WORKING,
EPISODIC, SEMANTIC) with RRF fusion. The first call loads the
cross-encoder lazily in a background thread; subsequent calls
reuse it.

In [ ]:
from arcane_retriever import ArcaneRetriever

retriever = ArcaneRetriever(sessions, session_dates=session_dates)

question = "When did the STDP fix ship?"
results = retriever.retrieve(question, qtype="temporal-reasoning", top_k=3)

for r in results:
    print(f"{r.rrf_score:5.2f}  {r.fact.text[:90]}")

## 4. Extract an answer with the rule-based extractor

`answer_extractor.extract_answer` is the no-LLM path. It uses
the Rust-accelerated regex extractor (~11× faster when the
crate is built) with a Python fallback.

In [ ]:
from answer_extractor import extract_answer

# Run the extractor on the top-ranked fact.
if results:
    text = results[0].fact.text
    print(f"Context: {text}")
    print(f"Question: {question}")
    print(f"Answer:   {extract_answer(question, text)!r}")
else:
    print("No results — check the decomposer output.")

## 5. Optional — run on the real LongMemEval oracle

The oracle is distributed separately and is gitignored in this
repo; see `docs/benchmarks/LongMemEval.md` for the acquisition
path. If `data/longmemeval_oracle.json` is present locally, the
next cell picks a random question and runs the same pipeline
on it. Otherwise the cell prints a pointer and returns cleanly.

In [ ]:
import json
import random
from pathlib import Path

from seed_utils import seed_everything

oracle_path = Path("../data/longmemeval_oracle.json")
if not oracle_path.exists():
    print("Oracle not present; see docs/benchmarks/LongMemEval.md.")
else:
    seed_everything(42)  # reproducible pick
    oracle = json.loads(oracle_path.read_text(encoding="utf-8"))
    item = random.choice(oracle)
    print(f"Question ({item['question_type']}): {item['question']}")
    print(f"Gold answer: {item['answer']}")
    print(f"#sessions: {len(item['haystack_sessions'])}")
    print()

    retriever = ArcaneRetriever(item["haystack_sessions"], session_dates=item["haystack_dates"])
    results = retriever.retrieve(item["question"], qtype=item["question_type"], top_k=3)
    for r in results:
        print(f"{r.rrf_score:5.2f}  {r.fact.text[:90]}")

## Next steps

- `docs/benchmarks/LongMemEval.md` — full benchmark methodology
- `bench_longmemeval.py --help` — run the bench end-to-end
- `docs/models/` — model cards for the five trained components
- `docs/adr/` — architectural decision records